# Recovering the table behind a published kappa

A study comparing two binary criteria on one population computes its numbers from a
2x2 table, then publishes the numbers and not the table. The quantity a reader wants —
how many cases the two criteria classify differently, and in which direction — is gone.
It is usually still recoverable, because the same reports print the sample size and both
marginal totals, and those pin the table down.

This notebook walks three cases: one the published digits determine exactly, one where a
coefficient of similar size belongs to an entirely different joint classification, and
one where the published figures admit no table at all. The same three cases appear in the
vignette of the R port, `enum2x2-r`, and reach the same numbers.

`enum2x2` itself needs only the standard library. The one plot below uses `matplotlib`.

In [ ]:
import enum2x2

enum2x2.__version__

## Cell names

The cells are named by which criterion is positive: `n11` positive on both, `n10`
positive on the first only, `n01` positive on the second only, `n00` negative on both.
The two positive marginals are `n_a = n11 + n10` and `n_b = n11 + n01`, and
`N = n11 + n10 + n01 + n00`.

The R port follows `metafor::conv.2x2` instead, so `n11`, `n10`, `n01`, `n00` there are
`ai`, `bi`, `ci`, `di`, and `n_a`, `n_b`, `N` are `n1i`, `n2i`, `ni`. The tables are the
same; only the labels differ.

## A determined case

A pooled series of 768 patients, two readings of the DSM-5 delirium criteria, published
at 60% raw agreement and kappa = 0.29. The strict reading is positive in 158 patients and
the relaxed reading in 466.

Figures are passed as the strings the source printed. `"0.29"` and `"0.290"` imply
different intervals and a float cannot tell them apart, so passing a float is refused.

In [ ]:
delirium = enum2x2.recover(768, n_a=158, n_b=466, kappa="0.29",
                           agreement="60", agreement_as_percent=True)
delirium

In [ ]:
delirium.status, len(delirium)

One table survives, so the report determines its own 2x2 exactly.

In [ ]:
t = delirium.table
t.as_dict()

In [ ]:
print(f"{'':10s} {'relaxed +':>10s} {'relaxed -':>10s}")
print(f"{'strict +':10s} {t.n11:>10d} {t.n10:>10d}")
print(f"{'strict -':10s} {t.n01:>10d} {t.n00:>10d}")

The two readings never cross-classify. No patient is strict-positive and
relaxed-negative, and every one of the patients they disagree on falls the same way.

In [ ]:
t.discordant          # (strict only, relaxed only)

In [ ]:
t.asymmetry           # 1.0 when the disagreement runs entirely one way

One reading is simply wider than the other, and its positives contain the other's.
Observed agreement is also the highest these two marginals permit, so 0.29 is the ceiling
here rather than a shortfall from it.

In [ ]:
{"published": t.kappa,
 "ceiling": enum2x2.kappa_max(158 / 768, 466 / 768),
 "floor": enum2x2.kappa_min(158 / 768, 466 / 768)}

None of that is in "kappa = 0.29", and kappa = 0.29 is what the paper printed.

## A coefficient of similar size, a different structure

A second comparison from the same corpus: 20,306 cases, the first criterion positive in
866 and the second in 1,603, published at kappa = 0.22. The coefficient is close to the
first one. The joint classification is not.

In [ ]:
cohort = enum2x2.recover(20306, n_a=866, n_b=1603, kappa="0.22")
cohort

Two decimals on a sample this size no longer determine the table, so the answer is a set
and the report is set-identified rather than unique. The span of each cell is what the
printed digits leave open.

In [ ]:
cohort.status, len(cohort)

In [ ]:
cohort.cell_ranges

Every table in the set has substantial counts in both off-diagonal cells, so here the two
criteria do cross-classify.

In [ ]:
asym = [tab.asymmetry for tab in cohort]
min(asym), max(asym)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([tab.n10 for tab in cohort], [tab.n01 for tab in cohort], "o")
ax.set_xlabel("criterion 1 only (n10)")
ax.set_ylabel("criterion 2 only (n01)")
ax.set_title("The tables kappa = 0.22 admits on 20,306 cases")
fig.tight_layout()

Set side by side, the two reports differ in the only thing a reader would use them for.

In [ ]:
rows = [("delirium, 768", delirium), ("cohort, 20306", cohort)]
print(f"{'case':16s} {'kappa':>6s} {'tables':>7s} {'one way':>12s} "
      f"{'other way':>12s} {'asymmetry':>12s}")
for name, rec in rows:
    r = rec.cell_ranges
    span = lambda c: (str(r[c][0]) if r[c][0] == r[c][1]
                      else f"{r[c][0]}-{r[c][1]}")
    a = sorted(tab.asymmetry for tab in rec)
    asym_txt = (f"{a[0]:.2f}" if a[0] == a[-1] else f"{a[0]:.2f}-{a[-1]:.2f}")
    print(f"{name:16s} {rec.published['kappa']:>6s} {len(rec):>7d} "
          f"{span('n10'):>12s} {span('n01'):>12s} {asym_txt:>12s}")

A published kappa near 0.2 to 0.3 is consistent with two criteria that never disagree in
one direction, and with two criteria whose disagreement is split roughly three to one.
The coefficient does not distinguish them; the recovered table does.

## A case with no table behind it

A third report: 370 patients, the first criterion positive in 165 and the second in 160,
published at kappa = 0.48 and 73% agreement.

In [ ]:
bad = enum2x2.recover(370, n_a=165, n_b=160, kappa="0.48",
                      agreement="73", agreement_as_percent=True)
bad.status, bad.reason

The status is `infeasible`, and it is kept apart from `insufficient`, which is what a
report gets when a required figure was never published at all.

In [ ]:
enum2x2.recover(370, n_a=165, n_b=160).status

The reason names the figure that excluded. Dropping the published agreement leaves a
table, so it is the agreement and the kappa that fail to cohere rather than the kappa and
the marginals.

In [ ]:
enum2x2.recover(370, n_a=165, n_b=160, kappa="0.48").table.as_dict()

What this licenses is narrow. It says the published figures admit no common table under
the definitions and rounding conventions used here. It does not say an error was made: a
denominator, an analysis set, or a version of a variable may simply have gone unstated,
and a weighted kappa or a different handling of missing cases would give different numbers
from the same study. The finding is that the printed figures cannot all be read off one
2x2, not that any of them is wrong.

A defect in the *call*, as opposed to a defect in the *source*, is raised rather than
reported.

In [ ]:
try:
    enum2x2.recover(100, n_a=101, n_b=50, kappa="0.5")
except enum2x2.InvalidInput as exc:
    print(type(exc).__name__, "-", exc)

try:
    enum2x2.recover(768, n_a=158, n_b=466, kappa=0.29)
except enum2x2.InvalidInput as exc:
    print(type(exc).__name__, "-", exc)

## A series of comparisons

`recover_many` takes a list of rows and never raises, so one malformed row does not stop
the rest. It carries a fifth status, `impossible`, for a row whose figures could not
describe any table; counting those as `insufficient` would inflate how many sources
under-reported.

In [ ]:
corpus = [
    {"study": "delirium pooled", "n": 768, "n_a": 158, "n_b": 466, "kappa": "0.29"},
    {"study": "cohort", "n": 20306, "n_a": 866, "n_b": 1603, "kappa": "0.22"},
    {"study": "third report", "n": 370, "n_a": 165, "n_b": 160, "kappa": "0.48"},
    {"study": "marginal above N", "n": 100, "n_a": 101, "n_b": 50, "kappa": "0.5"},
    {"study": "kappa never printed", "n": 768, "n_a": 158, "n_b": 466},
]
out = enum2x2.recover_many(corpus)

print(f"{'study':22s} {'status':14s} {'tables':>6s}  n11")
for row, rec in zip(corpus, out):
    r = rec.cell_ranges if rec.tables else None
    n11 = "-" if r is None else (str(r["n11"][0]) if r["n11"][0] == r["n11"][1]
                                 else f"{r['n11'][0]}-{r['n11'][1]}")
    print(f"{row['study']:22s} {rec.status:14s} {len(rec):>6d}  {n11}")

## Why rounding is the whole problem

With exact inputs the algebra is one line. For positive rates `p_A` and `p_B` on `N`
observations, expected agreement is `p_A*p_B + (1 - p_A)*(1 - p_B)`, observed agreement is
`kappa*(1 - p_e) + p_e`, and the both-positive cell is `(p_o + p_A + p_B - 1) / 2`.

Nobody prints exact inputs. A marginal printed `4.26%` and a kappa printed `0.22` are
intervals, and the trailing digits are information: `"0.1"` and `"0.10"` are different
inputs.

In [ ]:
(enum2x2.rounding_interval("0.1"),
 enum2x2.rounding_interval("0.10"),
 enum2x2.rounding_interval("4.26", as_percent=True))

In [ ]:
# The counts a printed marginal admits on a given sample.
from enum2x2._core import counts_rounding_to

counts_rounding_to("4.26", 20306, as_percent=True)

Membership is decided on the exact form of those intervals, against a table's statistic
written as a ratio of integers. Both sides are rational, so the comparison is exact and
there is no tolerance to choose.

In [ ]:
(enum2x2.exact_interval("0.22"), enum2x2.exact_kappa(158, 0, 308, 302))

The interval is closed at both ends. A value falling exactly on a boundary rounds up under
one convention and down under another, and sources do not say which they used. The table
(2, 0, 2, 20) on 24 cases has a kappa that is exactly one of those values, and it survives
either printing.

In [ ]:
k = enum2x2.exact_kappa(2, 0, 2, 20)
k, float(k)

In [ ]:
(enum2x2.recover(24, n_a=2, n_b=4, kappa="0.62").table,
 enum2x2.recover(24, n_a=2, n_b=4, kappa="0.63").table)

A half-open interval would lose the table under one of the two printings. Admitting both
ends can only widen the candidate set, which understates what the source identifies;
excluding one end can drop the true table, which asserts a precision the source does not
carry.

## What it will not do

It does not say which criterion is correct; neither does the table. It does not separate
disagreement *between* criteria from inconsistent *application* of either, which would
need the same criterion applied twice to the same cases. Three or more categories are out
of scope, since the identity above is the binary case.